# 🏗️ WORKFLOW COMPLET — Traitement Produits (PrestaShop / High-Tech Moving)

Ce notebook regroupe **tout le pipeline** en une suite de cellules exécutables, dans l'ordre :

1. Installation & détection de l'environnement (Colab ou Local)
2. **Cellule 1** — Upload + nettoyage + fusion des doublons (somme des quantités)
3. **Cellule 2** — Vérification/correction EAN (13 chiffres) + longueur des champs texte (≤128 caractères)
4. **Cellule 3** — Prédiction ML de `idMarqueSite` et `idCategorySite`
5. **Cellule 4** — Génération `description`, `recap`, `meta_title`, `meta_description`, `meta_keywords`, `mot_clés`
6. **Cellule 5** — Calcul `sku`, `price_particulier`, `price_profesionnel`, `id_tax_rules_group`
7. **Cellule 6** — Scraping des images produit (Bing) + nettoyage des URLs
8. **Cellule 7** — Assemblage final + export Excel
9. **Cellule TEST** — Vérification rapide sur un échantillon fourni (sans upload)

✅ Fonctionne **sur Google Colab** (upload via widget, téléchargement auto) et **en local** (saisie du chemin de fichier, sauvegarde dans `./output/`).


## 0) Installation des librairies & détection de l'environnement

In [1]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

def pip_install(pkgs):
    for p in pkgs:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
        except Exception as e:
            print(f"⚠️ Impossible d'installer {p} : {e}")

pip_install([
    "pandas", "openpyxl", "scikit-learn", "xgboost", "joblib",
    "selenium", "webdriver-manager",
])

print(f"🖥️ Environnement détecté : {'Google Colab' if IN_COLAB else 'Local (application data automation)'}")


🖥️ Environnement détecté : Google Colab


## 0bis) Fonctions universelles d'upload / export (Colab + Local)

In [2]:
import os, io, pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


def upload_fichier(message="Sélectionnez un fichier (.csv, .xlsx, .xls)"):
    """
    Upload universel :
      - Sur Colab   : widget d'upload natif.
      - En local    : saisie du chemin complet du fichier au clavier.
    Retourne (dataframe, nom_fichier).
    """
    if IN_COLAB:
        from google.colab import files
        print(message)
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("Aucun fichier n'a été sélectionné.")
        nom_fichier = list(uploaded.keys())[0]
        source = io.BytesIO(uploaded[nom_fichier])
    else:
        chemin = input(f"{message}\nChemin complet du fichier : ").strip().strip('"')
        if not os.path.isfile(chemin):
            raise FileNotFoundError(f"Fichier introuvable : {chemin}")
        nom_fichier = os.path.basename(chemin)
        source = chemin

    ext = os.path.splitext(nom_fichier)[1].lower()

    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(source)
    elif ext == ".csv":
        # IMPORTANT : on ne s'arrête pas à la 1ère combinaison séparateur/moteur qui ne
        # lève pas d'exception, car pd.read_csv peut "réussir" en mettant tout dans 1
        # seule colonne si le séparateur est faux (ex: sep=None mal détecté sur un
        # fichier en ';'). On teste chaque combinaison et on garde celle qui donne le
        # PLUS de colonnes.
        # NOTE : le moteur "c" (par défaut pandas) est essayé AVANT le moteur "python",
        # car ce dernier est plus strict sur les guillemets et échoue sur des champs
        # pourtant valides du type   wd red pro 3.5" 8tb...   que le moteur c gère
        # très bien.
        df = None
        tentatives = [
            (";", "c"), (",", "c"), ("\t", "c"),
            (";", "python"), (",", "python"), ("\t", "python"), (None, "python"),
        ]
        for sep, engine in tentatives:
            try:
                if isinstance(source, io.BytesIO):
                    source.seek(0)
                candidat = pd.read_csv(source, sep=sep, engine=engine)
            except Exception:
                continue
            if df is None or candidat.shape[1] > df.shape[1]:
                df = candidat
            if df.shape[1] > 1:
                break  # séparateur/moteur correct trouvé, inutile de tester le reste
        if df is None:
            raise ValueError("Impossible de lire le CSV (séparateurs testés : ';', ',', tab, auto).")
        if df.shape[1] == 1:
            print("⚠️ Le fichier CSV n'a été lu qu'avec 1 seule colonne quel que soit le "
                  "séparateur testé (';', ',', tab). Vérifiez le fichier source.")
    else:
        raise ValueError(f"Format non supporté : {ext}. Utilisez .csv, .xlsx ou .xls.")

    print(f"✅ Fichier chargé : {nom_fichier} — {df.shape[0]} lignes x {df.shape[1]} colonnes")
    return df, nom_fichier


def sauvegarder_et_telecharger(df, nom_fichier, dossier_local="output"):
    """
    Sauvegarde le DataFrame en .xlsx.
      - Sur Colab : déclenche le téléchargement automatique.
      - En local  : sauvegarde dans ./output/.
    """
    if not IN_COLAB:
        os.makedirs(dossier_local, exist_ok=True)
        nom_fichier = os.path.join(dossier_local, os.path.basename(nom_fichier))

    df.to_excel(nom_fichier, index=False)
    print(f"✅ Fichier sauvegardé : {nom_fichier}")

    if IN_COLAB:
        from google.colab import files
        files.download(nom_fichier)

    return nom_fichier


def trouver_colonne(nom_cherche, colonnes):
    """Recherche insensible à la casse/espaces d'un nom de colonne."""
    mapping = {c.lower().strip(): c for c in colonnes}
    return mapping.get(nom_cherche.lower().strip())


## 1) CELLULE 1 — Upload + Nettoyage + Fusion des doublons

- Détecte automatiquement les colonnes `NOM DU PRODUIT`, `EAN`, `etat`, `pc`, `prixVente`, `quantite`.
- Nettoie les montants (`€`, virgules).
- Fusionne les lignes identiques (même nom + EAN + état + pc + prixVente) en **additionnant les quantités**.


In [3]:
df_brut, nom_source = upload_fichier(
    "📂 Upload du fichier STOCK brut (colonnes attendues : NOM DU PRODUIT, EAN, etat, pc, prixVente, quantite)"
)

df_brut.columns = df_brut.columns.str.strip()

col_nom  = trouver_colonne("NOM DU PRODUIT", df_brut.columns) or trouver_colonne("name", df_brut.columns)
col_ean  = trouver_colonne("EAN", df_brut.columns) or trouver_colonne("ean", df_brut.columns)
col_etat = trouver_colonne("etat", df_brut.columns)
col_pc   = trouver_colonne("pc", df_brut.columns)
col_pv   = trouver_colonne("prixVente", df_brut.columns)
col_qte  = trouver_colonne("quantite", df_brut.columns)

for label, col in [("NOM DU PRODUIT", col_nom), ("EAN", col_ean), ("etat", col_etat),
                    ("pc", col_pc), ("prixVente", col_pv), ("quantite", col_qte)]:
    if col is None:
        raise ValueError(f"❌ Colonne obligatoire introuvable dans le fichier : {label}")

df = df_brut.rename(columns={
    col_nom: "NOM DU PRODUIT", col_ean: "EAN", col_etat: "etat",
    col_pc: "pc", col_pv: "prixVente", col_qte: "quantite",
})

# Nettoyage texte de base
df["NOM DU PRODUIT"] = df["NOM DU PRODUIT"].astype(str).str.strip()
df["etat"] = df["etat"].astype(str).str.strip().str.upper()

# Nettoyage des montants (virgule décimale, symbole €, espaces)
for col in ["pc", "prixVente"]:
    df[col] = (
        df[col].astype(str)
        .str.replace("€", "", regex=False)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["quantite"] = pd.to_numeric(df["quantite"], errors="coerce").fillna(0)

# Fusion des doublons : même produit → on additionne les quantités
df_clean = (
    df.groupby(["NOM DU PRODUIT", "EAN", "etat", "pc", "prixVente"], as_index=False, dropna=False)
      .agg({"quantite": "sum"})
)

print("\n✅ Nettoyage terminé")
print(f"   Lignes avant        : {len(df)}")
print(f"   Lignes après fusion : {len(df_clean)}")
print(f"   Doublons fusionnés  : {len(df) - len(df_clean)}")

display(df_clean.head(10))


📂 Upload du fichier STOCK brut (colonnes attendues : NOM DU PRODUIT, EAN, etat, pc, prixVente, quantite)


Saving test prix quantity dataset.xlsx to test prix quantity dataset (1).xlsx
✅ Fichier chargé : test prix quantity dataset (1).xlsx — 50 lignes x 8 colonnes

✅ Nettoyage terminé
   Lignes avant        : 50
   Lignes après fusion : 45
   Doublons fusionnés  : 5


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite
0,12 Pro Silicone Case with MagSafe - White,194252169094,NOP,59.0,59.0,4
1,Amiibo supersmah bros terry,45496893767,NSB,-1.0,35.0,22
2,Amiibo supersmash bros banjo & kazooie,45496893750,NSB,35.0,35.0,0
3,Amiibo supersmash bros joker,45496594183,NSB,35.0,35.0,-1
4,Apple Clear Case with MagSafe for iPhone 12/12...,194252169575,NOP,59.0,59.0,10
5,Apple Coque Leather MagSafe iPhone 12 (Pro) - ...,194252168257,NOP,69.0,69.0,2
6,Apple Leather Case (for iPhone SE) - Midnight ...,190199610552,NOP,59.0,59.0,2
7,Apple Leather Case MIDNIGHT BLUE iPhone 7 Plus,190198005557,NOP,59.0,59.0,1
8,Apple Leather Case for iPhone 7 Plus - Geranium,190198447371,NOP,59.0,59.0,3
9,Apple Leather Case for iPhone 7 Plus - Taupe,190198357441,NOP,59.0,59.0,1


## 1bis) CELLULE 1bis — Contrôle qualité : `quantite`, `pc`, `prixVente` (≤ 0 ou vides)

Cette étape s'exécute **juste après la fusion des doublons**. Elle sépare :
- les lignes correctes (`quantite`, `pc` et `prixVente` tous strictement positifs et non vides) → elles continuent dans le pipeline (`df_clean`) ;
- les lignes en erreur (valeur ≤ 0, vide, ou non numérique) → exportées à part dans `lignes_avec_erreurs.xlsx` pour vérification manuelle, et retirées de la suite du traitement.

In [4]:
import numpy as np

COLONNES_A_CONTROLER = ["quantite", "pc", "prixVente"]

for col in COLONNES_A_CONTROLER:
    if col not in df_clean.columns:
        raise ValueError(f"❌ La colonne '{col}' est introuvable dans df_clean.")


def parser_valeur_numerique(valeur):
    """Convertit une valeur en float, en gérant virgules décimales et espaces."""
    if pd.isna(valeur):
        return None, False
    if isinstance(valeur, (int, float, np.integer, np.floating)):
        return float(valeur), True
    texte = str(valeur).strip().replace("\u00a0", "").replace(" ", "")
    if texte == "" or texte.lower() in ["nan", "none", "null", "-"]:
        return None, False
    if "," in texte and "." not in texte:
        texte = texte.replace(",", ".")
    elif "," in texte and "." in texte:
        texte = texte.replace(",", "")
    try:
        return float(texte), True
    except ValueError:
        return None, False


def controler_colonne(valeur, nom_colonne):
    """Retourne un message d'erreur, ou None si la valeur est correcte."""
    if pd.isna(valeur):
        return f"{nom_colonne} vide"
    val, succes = parser_valeur_numerique(valeur)
    if not succes:
        return f"{nom_colonne} non numérique"
    if val < 0:
        return f"{nom_colonne} négatif"
    if val == 0:
        return f"{nom_colonne} = 0"
    return None


df_controle = df_clean.copy()

for col in COLONNES_A_CONTROLER:
    df_controle[f"_erreur_{col}"] = df_controle[col].apply(
        lambda v, c=col: controler_colonne(v, c)
    )


def construire_message_erreur(row):
    erreurs = [
        row[f"_erreur_{col}"] for col in COLONNES_A_CONTROLER
        if pd.notna(row[f"_erreur_{col}"])
    ]
    return " ; ".join(erreurs) if erreurs else None


df_controle["Erreur"] = df_controle.apply(construire_message_erreur, axis=1)
masque_erreur = df_controle["Erreur"].notna()

df_lignes_correctes = df_clean.loc[~masque_erreur].copy()
df_lignes_erreurs = df_clean.loc[masque_erreur].copy()
df_lignes_erreurs["Erreur"] = df_controle.loc[masque_erreur, "Erreur"].values

total = len(df_clean)
nb_correctes = len(df_lignes_correctes)
nb_erreurs = len(df_lignes_erreurs)

print("===== CONTRÔLE QUALITÉ : quantite / pc / prixVente =====")
print(f"Total des lignes    : {total}")
print(f"Lignes correctes    : {nb_correctes}")
print(f"Lignes en erreur    : {nb_erreurs}")
if total:
    print(f"Taux de conformité  : {nb_correctes/total*100:.2f}%")
    print(f"Taux d'erreur       : {nb_erreurs/total*100:.2f}%")

if nb_erreurs > 0:
    print("\n📋 Répartition des erreurs :")
    print(df_lignes_erreurs["Erreur"].value_counts())
    sauvegarder_et_telecharger(df_lignes_erreurs, "lignes_avec_erreurs.xlsx")
else:
    print("\n✅ Aucune ligne en erreur détectée.")

# On ne garde que les lignes correctes pour la suite du pipeline
df_clean = df_lignes_correctes.reset_index(drop=True)

print(f"\n➡️  {len(df_clean)} lignes valides transmises à la suite du pipeline.")
display(df_clean.head(10))


===== CONTRÔLE QUALITÉ : quantite / pc / prixVente =====
Total des lignes    : 45
Lignes correctes    : 39
Lignes en erreur    : 6
Taux de conformité  : 86.67%
Taux d'erreur       : 13.33%

📋 Répartition des erreurs :
Erreur
pc négatif           1
quantite = 0         1
quantite négatif     1
pc = 0               1
prixVente = 0        1
prixVente négatif    1
Name: count, dtype: int64
✅ Fichier sauvegardé : lignes_avec_erreurs.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


➡️  39 lignes valides transmises à la suite du pipeline.


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite
0,12 Pro Silicone Case with MagSafe - White,194252169094,NOP,59.0,59.0,4
1,Apple Clear Case with MagSafe for iPhone 12/12...,194252169575,NOP,59.0,59.0,10
2,Apple Coque Leather MagSafe iPhone 12 (Pro) - ...,194252168257,NOP,69.0,69.0,2
3,Apple Leather Case (for iPhone SE) - Midnight ...,190199610552,NOP,59.0,59.0,2
4,Apple Leather Case MIDNIGHT BLUE iPhone 7 Plus,190198005557,NOP,59.0,59.0,1
5,Apple Leather Case for iPhone 7 Plus - Geranium,190198447371,NOP,59.0,59.0,3
6,Apple Leather Case for iPhone 7 Plus - Taupe,190198357441,NOP,59.0,59.0,1
7,Apple iPhone 7 Plus Leather Case Berry,190198376404,NOP,59.0,59.0,1
8,COQUE iPhone 12/12 Pro avec MagSafe (Deep Navy),194252169056,NOP,59.0,59.0,1
9,Cas Cuir Apple iPhone 12 Pro Max Bleu Baltique,194252168455,NOP,69.0,69.0,2


## 1ter) CELLULE 1ter — Comparaison avec le backoffice existant (produits déjà en ligne vs nouveaux produits)

Cette étape s'exécute **juste après le contrôle qualité** (quantite/pc/prixVente), lui-même exécuté juste après la fusion des doublons. Elle compare `df_clean` (issu du fichier stock nettoyé) avec un export du **backoffice** (colonnes `name` et `ean13`) sur le couple **nom produit + EAN** :
- les produits **déjà présents** dans le backoffice → exportés dans `produits_dupliques.xlsx` (mis de côté, pas retraités) ;
- les produits **absents du backoffice** → ce sont les **nouveaux produits à créer**, exportés dans `produits_non_dupliques.xlsx` et transmis à la suite du pipeline (contrôle qualité, mapping état, description, SKU, images, etc.).

In [5]:
# Upload du fichier BACKOFFICE (export des produits déjà en ligne sur le site)
df_backoffice, nom_backoffice = upload_fichier(
    "📂 Upload du fichier BACKOFFICE existant (colonnes attendues : name, ean13)"
)

df_backoffice.columns = df_backoffice.columns.str.strip()

col_name_back = trouver_colonne("name", df_backoffice.columns) or trouver_colonne("NOM DU PRODUIT", df_backoffice.columns)
col_ean_back  = trouver_colonne("ean13", df_backoffice.columns) or trouver_colonne("EAN", df_backoffice.columns)

if col_name_back is None or col_ean_back is None:
    raise ValueError(
        "❌ Colonnes 'name' / 'ean13' introuvables dans le fichier backoffice. "
        f"Colonnes disponibles : {list(df_backoffice.columns)}"
    )

df_backoffice = df_backoffice.rename(columns={col_name_back: "name", col_ean_back: "ean13"})

# --- Nettoyage des clés de comparaison (même logique des deux côtés) ---
df_clean["EAN"] = df_clean["EAN"].astype(str).str.replace(".0", "", regex=False).str.strip()
df_clean["_nom_comp"] = df_clean["NOM DU PRODUIT"].astype(str).str.lower().str.strip()

df_backoffice["ean13"] = df_backoffice["ean13"].astype(str).str.replace(".0", "", regex=False).str.strip()
df_backoffice["_nom_comp"] = df_backoffice["name"].astype(str).str.lower().str.strip()

# On dé-doublonne le backoffice sur la clé de comparaison pour ne pas multiplier
# les lignes de df_clean pendant le merge si le backoffice contient des doublons.
df_backoffice_cle = df_backoffice[["_nom_comp", "ean13"]].drop_duplicates()

# --- Comparaison sur NOM DU PRODUIT (en minuscule) + EAN ---
df_merge = df_clean.merge(
    df_backoffice_cle,
    left_on=["_nom_comp", "EAN"],
    right_on=["_nom_comp", "ean13"],
    how="left",
    indicator=True,
)

colonnes_originales = [c for c in df_clean.columns if c != "_nom_comp"]

produits_dupliques = df_merge.loc[df_merge["_merge"] == "both", colonnes_originales].copy()
produits_nouveaux  = df_merge.loc[df_merge["_merge"] == "left_only", colonnes_originales].copy()

print("===== COMPARAISON AVEC LE BACKOFFICE =====")
print(f"Produits déjà en ligne (dupliqués) : {len(produits_dupliques)}")
print(f"Produits nouveaux à créer          : {len(produits_nouveaux)}")

if len(produits_dupliques) > 0:
    sauvegarder_et_telecharger(produits_dupliques, "produits_dupliques.xlsx")
else:
    print("ℹ️ Aucun produit dupliqué avec le backoffice.")

sauvegarder_et_telecharger(produits_nouveaux, "produits_non_dupliques.xlsx")

# ➡️ Seuls les nouveaux produits (absents du backoffice) continuent dans le pipeline
df_clean = produits_nouveaux.reset_index(drop=True)

print(f"\n➡️  {len(df_clean)} nouveaux produits transmis à la suite du pipeline "
      "(contrôle qualité, mapping état, description, SKU, images...).")
display(df_clean.head(10))


📂 Upload du fichier BACKOFFICE existant (colonnes attendues : name, ean13)


Saving data back office (2).csv to data back office (2) (1).csv
✅ Fichier chargé : data back office (2) (1).csv — 2051 lignes x 23 colonnes
===== COMPARAISON AVEC LE BACKOFFICE =====
Produits déjà en ligne (dupliqués) : 33
Produits nouveaux à créer          : 6
✅ Fichier sauvegardé : produits_dupliques.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Fichier sauvegardé : produits_non_dupliques.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


➡️  6 nouveaux produits transmis à la suite du pipeline (contrôle qualité, mapping état, description, SKU, images...).


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite
0,Apple Clear Case with MagSafe for iPhone 12/12...,194252169575,NOP,59.0,59.0,10
1,Coque silicone Apple iPhone SE bleu abys,194253035466,NOP,59.0,59.0,2
2,Cque silicone Apple iPhone SE rose craie,194253035497,NOP,59.0,59.0,1
3,IPHONE 12/12 PRO case,194252168295,NOP,59.0,59.0,1
4,IPHONE SE case,194253035497,NOP,39.0,39.0,1
5,iphone 12/12pro silicone case deep navy,194252169056,NOP,59.0,59.0,1


## 1quater) CELLULE 1quater — Mapping `etat` → valeurs PrestaShop (Neuf / Comme neuf / Occasion / reconditionné)

**Correction** : ce mapping doit être fait ici, **avant** la génération de `description`, `recap` et des champs `meta_*` (Cellule 4), sinon ces colonnes récupèrent le code brut (ex: `NSB`) au lieu de la valeur finale (ex: `Neuf`).

Le code brut d'origine est conservé dans `etat_brut` et sera réutilisé pour construire le SKU (Cellule 5).

In [6]:
CONDITION_MAPPING = {
    # ── NEUF ──
    "N":            "Neuf",
    "N/SANS BOITE": "Neuf",
    "NSB":          "Neuf",
    "NEUF":         "Neuf",
    "DEFAULT":      "Neuf",

    # ── COMME NEUF ──
    "NOP":          "Comme neuf",
    "NWB":          "Comme neuf",
    "CN":           "Comme neuf",
    "USEDCN":       "Comme neuf",

    # ── OCCASION ──
    "USEDA":        "Occasion",
    "USEDB":        "Occasion",
    "USEBC":        "Occasion",
    "USEDC":        "Occasion",
    "USEDBC":       "Occasion",
    "OBESB":        "Occasion",
    "OAP":          "Occasion",
    "O":            "Occasion",
    "OCCASION":     "Occasion",

    # ── RECONDITIONNÉ ──
    "RECNEUF":      "reconditionné",
    "RECMINT":      "reconditionné",
    "RECTBE":       "reconditionné",
    "RECBE":        "reconditionné",
    "RECEC":        "reconditionné",
}
CONDITION_DEFAUT = "Neuf"  # valeur appliquée si le code n'est reconnu dans aucune catégorie


def mapper_condition(valeur):
    code = str(valeur).strip().upper()
    return CONDITION_MAPPING.get(code, CONDITION_DEFAUT)


# On garde le code brut d'origine pour le SKU (ex: NSB-0850032692168)
df_clean["etat_brut"] = df_clean["etat"].astype(str).str.strip().str.upper()

etats_non_reconnus = sorted(set(df_clean["etat_brut"]) - set(CONDITION_MAPPING.keys()))
if etats_non_reconnus:
    print(f"⚠️ Codes état non reconnus (mappés par défaut sur '{CONDITION_DEFAUT}') : {etats_non_reconnus}")

df_clean["etat"] = df_clean["etat_brut"].apply(mapper_condition)

print("✅ Colonne 'etat' mappée vers les valeurs PrestaShop (Neuf / Comme neuf / Occasion / reconditionné).")
display(df_clean[["NOM DU PRODUIT", "etat_brut", "etat"]].head(10))


✅ Colonne 'etat' mappée vers les valeurs PrestaShop (Neuf / Comme neuf / Occasion / reconditionné).


,NOM DU PRODUIT,etat_brut,etat
0,Apple Clear Case with MagSafe for iPhone 12/12...,NOP,Comme neuf
1,Coque silicone Apple iPhone SE bleu abys,NOP,Comme neuf
2,Cque silicone Apple iPhone SE rose craie,NOP,Comme neuf
3,IPHONE 12/12 PRO case,NOP,Comme neuf
4,IPHONE SE case,NOP,Comme neuf
5,iphone 12/12pro silicone case deep navy,NOP,Comme neuf


## 2) CELLULE 2 — Vérification / correction EAN (13 chiffres) + longueur des champs (≤128)

- **EAN** : complété à 13 chiffres avec des zéros si trop court ; signalé (non tronqué) si trop long.
- **Champs texte** (`NOM DU PRODUIT`, et plus tard `meta_title`, `meta_description`, `meta_keywords`, `mot_clés`) : tronqués à 128 caractères.


In [7]:
def corriger_ean(df, colonne="EAN", longueur_cible=13):
    """Complète les EAN trop courts avec des zéros ; signale ceux trop longs (non modifiés)."""
    df = df.copy()
    nb_corriges, nb_anomalies, anomalies = 0, 0, []

    def corriger(valeur):
        nonlocal nb_corriges, nb_anomalies
        if pd.isna(valeur):
            return valeur
        texte = str(valeur).strip()
        if texte.endswith(".0"):
            texte = texte[:-2]
        chiffres = "".join(c for c in texte if c.isdigit())
        if len(chiffres) == 0:
            return valeur
        if len(chiffres) < longueur_cible:
            nb_corriges += 1
            return chiffres.zfill(longueur_cible)
        elif len(chiffres) > longueur_cible:
            nb_anomalies += 1
            anomalies.append(chiffres)
            return chiffres
        return chiffres

    df[colonne] = df[colonne].apply(corriger)
    print(f"✅ EAN — complétés à 13 chiffres : {nb_corriges} | anomalies (>13 chiffres, à vérifier) : {nb_anomalies}")
    return df, nb_corriges, nb_anomalies, anomalies


def nettoyer_longueur_champs(df, colonnes=None, max_longueur=128):
    """Tronque à max_longueur caractères les colonnes texte présentes dans le DataFrame."""
    df = df.copy()
    if colonnes is None:
        colonnes = ["NOM DU PRODUIT", "name", "meta_title", "meta_description", "meta_keywords", "mot_clés"]
    colonnes_presentes = [c for c in colonnes if c in df.columns]
    compteur = {c: 0 for c in colonnes_presentes}

    for col in colonnes_presentes:
        def tronquer(valeur, col=col):
            if pd.isna(valeur):
                return valeur
            texte = str(valeur)
            if len(texte) > max_longueur:
                compteur[col] += 1
                return texte[:max_longueur]
            return texte
        df[col] = df[col].apply(tronquer)

    print(f"✅ Longueur des champs (max {max_longueur} caractères) :")
    for col, nb in compteur.items():
        print(f"   - {col} : {nb} valeur(s) tronquée(s)")
    return df, compteur


df_clean, nb_ean_corriges, nb_ean_anomalies, liste_anomalies_ean = corriger_ean(df_clean, colonne="EAN")
df_clean, stats_longueur = nettoyer_longueur_champs(df_clean, colonnes=["NOM DU PRODUIT"])

if liste_anomalies_ean:
    print(f"\n⚠️ {len(liste_anomalies_ean)} EAN à vérifier manuellement (plus de 13 chiffres) :")
    print(liste_anomalies_ean[:10])

display(df_clean.head(10))


✅ EAN — complétés à 13 chiffres : 6 | anomalies (>13 chiffres, à vérifier) : 0
✅ Longueur des champs (max 128 caractères) :
   - NOM DU PRODUIT : 0 valeur(s) tronquée(s)


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite,etat_brut
0,Apple Clear Case with MagSafe for iPhone 12/12...,0194252169575,Comme neuf,59.0,59.0,10,NOP
1,Coque silicone Apple iPhone SE bleu abys,0194253035466,Comme neuf,59.0,59.0,2,NOP
2,Cque silicone Apple iPhone SE rose craie,0194253035497,Comme neuf,59.0,59.0,1,NOP
3,IPHONE 12/12 PRO case,0194252168295,Comme neuf,59.0,59.0,1,NOP
4,IPHONE SE case,0194253035497,Comme neuf,39.0,39.0,1,NOP
5,iphone 12/12pro silicone case deep navy,0194252169056,Comme neuf,59.0,59.0,1,NOP


## 3) CELLULE 3 — Prédiction ML de `idMarqueSite` et `idCategorySite`

Nécessite un fichier **historique** (produits déjà classés) avec au minimum les colonnes :
`name`, `id_marque`, `id_category`.

Plusieurs modèles sont testés (SVC, Logistic Regression, Naive Bayes, SGD, Random Forest, KNN, XGBoost),
le meilleur (accuracy la plus haute) est conservé pour prédire sur le dataset courant.

➡️ Mettez `ACTIVER_PREDICTION_ML = False` si vous n'avez pas de fichier historique : les colonnes seront mises à `0`.


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

ACTIVER_PREDICTION_ML = True  # -> False si pas de fichier historique disponible


def creer_modeles():
    modeles = {
        "SVC": LinearSVC(),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Naive Bayes": MultinomialNB(),
        "SGD Classifier": SGDClassifier(loss="hinge", max_iter=1000),
        # "KNN": KNeighborsClassifier(n_neighbors=5),
    }
    if HAS_XGB:
        modeles["XGBoost"] = XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            random_state=42, eval_metric="mlogloss",
        )
    return modeles


def pipeline_tfidf(modele):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3), min_df=1)),
        ("model", modele),
    ])


def entrainer_meilleur_modele(X, y):
    """Entraîne tous les modèles et retourne (pipeline_entraîné, label_encoder, nom_meilleur, accuracy)."""
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))

    scores, pipelines = {}, {}
    for nom, modele in creer_modeles().items():
        try:
            pipe = pipeline_tfidf(modele)
            pipe.fit(X, y_enc)
            pred = pipe.predict(X)
            acc = accuracy_score(y_enc, pred)
            scores[nom] = acc
            pipelines[nom] = pipe
            print(f"   {nom:22s} → accuracy {acc*100:.2f}%")
        except Exception as e:
            print(f"   ⚠️ {nom} a échoué : {e}")

    if not scores:
        raise RuntimeError("Aucun modèle n'a pu être entraîné.")

    nom_best = max(scores, key=scores.get)
    print(f"\n🏆 Meilleur modèle : {nom_best} ({scores[nom_best]*100:.2f}%)")
    return pipelines[nom_best], le, nom_best, scores[nom_best]


if ACTIVER_PREDICTION_ML:
    df_hist, _ = upload_fichier("📂 Upload du fichier HISTORIQUE (colonnes : name, id_marque, id_category)")
    df_hist.columns = df_hist.columns.str.strip()

    for c in ["name", "id_marque", "id_category"]:
        if c not in df_hist.columns:
            raise ValueError(f"❌ Colonne obligatoire manquante dans l'historique : {c}")

    X_hist = df_hist["name"].astype(str)

    print("\n🎯 Entraînement — prédiction id_marque")
    modele_marque, le_marque, nom_modele_marque, acc_marque = entrainer_meilleur_modele(
        X_hist, df_hist["id_marque"]
    )

    print("\n🎯 Entraînement — prédiction id_category")
    modele_categorie, le_categorie, nom_modele_categorie, acc_categorie = entrainer_meilleur_modele(
        X_hist, df_hist["id_category"]
    )

    pred_marque_enc = modele_marque.predict(df_clean["NOM DU PRODUIT"].astype(str))
    pred_cat_enc    = modele_categorie.predict(df_clean["NOM DU PRODUIT"].astype(str))

    df_clean["idMarqueSite"]   = le_marque.inverse_transform(pred_marque_enc)
    df_clean["idCategorySite"] = le_categorie.inverse_transform(pred_cat_enc)

    print(f"\n✅ Prédictions ajoutées (modèles retenus : {nom_modele_marque} / {nom_modele_categorie})")
else:
    df_clean["idMarqueSite"] = 0
    df_clean["idCategorySite"] = 0
    print("⚠️ Prédiction ML désactivée — idMarqueSite/idCategorySite mis à 0 par défaut.")

display(df_clean.head(10))


📂 Upload du fichier HISTORIQUE (colonnes : name, id_marque, id_category)


Saving test id marque et id categorie.csv to test id marque et id categorie.csv
✅ Fichier chargé : test id marque et id categorie.csv — 4428 lignes x 3 colonnes

🎯 Entraînement — prédiction id_marque
   SVC                    → accuracy 99.77%
   Logistic Regression    → accuracy 86.07%
   Naive Bayes            → accuracy 75.93%
   SGD Classifier         → accuracy 99.75%
   Random Forest          → accuracy 99.77%
   KNN                    → accuracy 88.57%
   XGBoost                → accuracy 88.14%

🏆 Meilleur modèle : SVC (99.77%)

🎯 Entraînement — prédiction id_category
   SVC                    → accuracy 99.14%
   Logistic Regression    → accuracy 82.84%
   Naive Bayes            → accuracy 75.02%
   SGD Classifier         → accuracy 99.01%
   Random Forest          → accuracy 99.23%
   KNN                    → accuracy 87.06%
   XGBoost                → accuracy 89.70%

🏆 Meilleur modèle : Random Forest (99.23%)

✅ Prédictions ajoutées (modèles retenus : SVC / Random Forest)


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite,etat_brut,idMarqueSite,idCategorySite
0,Apple Clear Case with MagSafe for iPhone 12/12...,0194252169575,Comme neuf,59.0,59.0,10,NOP,35,1889
1,Coque silicone Apple iPhone SE bleu abys,0194253035466,Comme neuf,59.0,59.0,2,NOP,35,1889
2,Cque silicone Apple iPhone SE rose craie,0194253035497,Comme neuf,59.0,59.0,1,NOP,35,1889
3,IPHONE 12/12 PRO case,0194252168295,Comme neuf,59.0,59.0,1,NOP,35,316
4,IPHONE SE case,0194253035497,Comme neuf,39.0,39.0,1,NOP,35,1889
5,iphone 12/12pro silicone case deep navy,0194252169056,Comme neuf,59.0,59.0,1,NOP,35,1889


## 4) CELLULE 4 — Génération `description`, `recap` et champs `meta_*`

- Utilise le **template fixe** High-Tech Moving (adresse, contact, horaires…) avec une accroche variable selon le produit.
- Génère `meta_title`, `meta_description`, `meta_keywords`, `mot_clés`, tous **plafonnés à 128 caractères**.
- ⚡ Version 100% règles (pas de modèle IA lourd à télécharger) : rapide, fonctionne sur Colab **et en local**, sans GPU.


In [10]:
import random

DESCRIPTION_TEMPLATE = """<p><strong>Bonjour et bienvenue chez High-Tech Moving !</strong></p>
<p>{ACCROCHE}</p>
<p>🆕 <strong>État :</strong> {ETAT}</p>
<p>📜 Facture incluse<br>🛡️ Garantie constructeur</p>

<p>🌐 Plus de produits sur : <strong>www.hightechmoving.com</strong></p>
<p>📞 Contact : +33 6 11 87 70 10 / +33 1 71 60 53 88</p>

<h3>✅ Pourquoi choisir High-Tech Moving ?</h3>
<ul>
<li>🏅 10 ans d'expertise</li>
<li>🧾 Produits authentiques</li>
<li>💸 Tarifs imbattables</li>
<li>🤝 Service client réactif</li>
</ul>

<h3>📍 Où nous trouver ?</h3>
<p><strong>High-Tech Moving</strong><br>7B Rue Michel Chasles, 75012 Paris</p>

<h4>🚇 Transports :</h4>
<ul>
<li>Métros : 1 et 14</li>
<li>RER : A et D</li>
<li>Bus : 57 et 56</li>
</ul>

<p>🕒 Ouvert tous les jours de 10h à 20h</p>
<p><strong>⚡ Stock limité – Profitez de cette offre !</strong></p>"""

RECAP_TEMPLATE = (
    '''<table style="border:1px solid #ddd;border-collapse:collapse;width:100%;font-family:Arial,Helvetica,sans-serif;">'''
    '''<thead><tr><th style="padding:8px;background:#f2f2f2;text-align:left;">Attribut</th>'''
    '''<th style="padding:8px;background:#f2f2f2;text-align:left;">Valeur</th></tr></thead><tbody>'''
    '''<tr><td style="padding:8px;font-weight:bold;">Nom</td><td style="padding:8px;">{NOM}</td></tr>'''
    '''<tr><td style="padding:8px;font-weight:bold;">État</td><td style="padding:8px;">{ETAT}</td></tr>'''
    '''<tr><td style="padding:8px;font-weight:bold;">EAN</td><td style="padding:8px;">{EAN}</td></tr>'''
    '''<tr><td style="padding:8px;font-weight:bold;">Prix</td><td style="padding:8px;">{PRIX} €</td></tr>'''
    '''</tbody></table>'''
)

ACCROCHES = [
    "Nous vous proposons en liquidation exceptionnelle : <strong>{NOM}</strong>.",
    "Découvrez dès maintenant <strong>{NOM}</strong>, disponible en stock limité.",
    "Profitez d'une offre exceptionnelle sur <strong>{NOM}</strong>.",
    "En exclusivité chez High-Tech Moving : <strong>{NOM}</strong>.",
]


def generer_accroche(nom_produit, seed=None):
    rnd = random.Random(seed if seed is not None else abs(hash(nom_produit)) % (2**32))
    return rnd.choice(ACCROCHES).format(NOM=nom_produit)


def formatter_prix(p):
    try:
        return f"{float(p):.2f}"
    except Exception:
        return "0.00"


def construire_description(nom, etat, seed=None):
    return DESCRIPTION_TEMPLATE.format(ACCROCHE=generer_accroche(nom, seed), ETAT=etat)


def construire_recap(nom, etat, ean, prix):
    return RECAP_TEMPLATE.format(NOM=nom, ETAT=etat, EAN=ean, PRIX=formatter_prix(prix))


def construire_mots_cles(nom, max_len=128):
    texte = ", ".join(str(nom).lower().split())
    return texte[:max_len]


descriptions, recaps, meta_titles, meta_descriptions, meta_keywords, mots_cles_list = [], [], [], [], [], []

for _, row in df_clean.iterrows():
    nom = str(row["NOM DU PRODUIT"])
    etat_txt = str(row["etat"]).capitalize()
    ean = str(row["EAN"])
    prix = row["prixVente"]

    descriptions.append(construire_description(nom, etat_txt, seed=row["EAN"]))
    recaps.append(construire_recap(nom, etat_txt, ean, prix))

    meta_titles.append(nom[:128])
    meta_descriptions.append(f"{nom} - {etat_txt}"[:128])
    mots = construire_mots_cles(nom)
    meta_keywords.append(mots)
    mots_cles_list.append(mots)

df_clean["description"]      = descriptions
df_clean["recap"]            = recaps
df_clean["meta_title"]       = meta_titles
df_clean["meta_description"] = meta_descriptions
df_clean["meta_keywords"]    = meta_keywords
df_clean["mot_clés"]         = mots_cles_list

# Sécurité : re-vérification de la longueur des champs générés
df_clean, _ = nettoyer_longueur_champs(
    df_clean, colonnes=["meta_title", "meta_description", "meta_keywords", "mot_clés"]
)

print("✅ description, recap et champs meta générés pour tous les produits.")
display(df_clean[["NOM DU PRODUIT", "meta_title", "meta_description"]].head())


✅ Longueur des champs (max 128 caractères) :
   - meta_title : 0 valeur(s) tronquée(s)
   - meta_description : 0 valeur(s) tronquée(s)
   - meta_keywords : 0 valeur(s) tronquée(s)
   - mot_clés : 0 valeur(s) tronquée(s)
✅ description, recap et champs meta générés pour tous les produits.


,NOM DU PRODUIT,meta_title,meta_description
0,Apple Clear Case with MagSafe for iPhone 12/12...,Apple Clear Case with MagSafe for iPhone 12/12...,Apple Clear Case with MagSafe for iPhone 12/12...
1,Coque silicone Apple iPhone SE bleu abys,Coque silicone Apple iPhone SE bleu abys,Coque silicone Apple iPhone SE bleu abys - Com...
2,Cque silicone Apple iPhone SE rose craie,Cque silicone Apple iPhone SE rose craie,Cque silicone Apple iPhone SE rose craie - Com...
3,IPHONE 12/12 PRO case,IPHONE 12/12 PRO case,IPHONE 12/12 PRO case - Comme neuf
4,IPHONE SE case,IPHONE SE case,IPHONE SE case - Comme neuf


## 5) CELLULE 5 — SKU, prix HT (particulier / professionnel), TVA

- `sku` = `etat` + `-` + `EAN`.
- `price_particulier` / `price_profesionnel` = prix TTC ÷ 1.20 (arrondi à 2 décimales) — cohérent avec l'exemple fourni (ex. 99.00 € → 82.5).
- `id_tax_rules_group` = 70 par défaut (ajustez si vous avez une correspondance catégorie → groupe de taxe).


In [ ]:
# ── SKU (basé sur le code état brut, ex: NSB-0850032692168) ──
# NOTE : le mapping de l'état vers les valeurs PrestaShop (Neuf / Comme neuf / ...)
# est déjà fait plus haut (Cellule 1ter), AVANT la génération des textes.
# On utilise donc ici "etat_brut" (code d'origine, ex: NSB) pour le SKU,
# et "etat" contient déjà la valeur finale PrestaShop.
df_clean["sku"] = df_clean["etat_brut"].astype(str) + "-" + df_clean["EAN"].astype(str)

TAUX_TVA = 1.20
df_clean["price_particulier"]  = (df_clean["prixVente"] / TAUX_TVA).round(2)
df_clean["price_profesionnel"] = (df_clean["prixVente"] / TAUX_TVA).round(2)

df_clean["id_tax_rules_group"] = 70
df_clean["actif"] = 1
print("✅ SKU et prix HT calculés.")
display(df_clean[["NOM DU PRODUIT", "EAN", "etat_brut", "etat", "sku", "prixVente", "price_particulier","actif"]].head())


✅ SKU et prix HT calculés.


,NOM DU PRODUIT,EAN,etat_brut,etat,sku,prixVente,price_particulier,actif
0,Apple Clear Case with MagSafe for iPhone 12/12...,0194252169575,NOP,Comme neuf,NOP-0194252169575,59.0,49.17,1
1,Coque silicone Apple iPhone SE bleu abys,0194253035466,NOP,Comme neuf,NOP-0194253035466,59.0,49.17,1
2,Cque silicone Apple iPhone SE rose craie,0194253035497,NOP,Comme neuf,NOP-0194253035497,59.0,49.17,1
3,IPHONE 12/12 PRO case,0194252168295,NOP,Comme neuf,NOP-0194252168295,59.0,49.17,1
4,IPHONE SE case,0194253035497,NOP,Comme neuf,NOP-0194253035497,39.0,32.50,1


## 6) CELLULE 6 — Scraping des images produits (Bing) + nettoyage des URLs

Combine en une seule étape :
1. La recherche d'images via Selenium (nom du produit → Bing Images),
2. Le nettoyage des URLs brutes (extraction des `"murl"`, suppression des paramètres inutiles, filtrage des domaines à éviter).

➡️ Mettez `ACTIVER_SCRAPING_IMAGES = False` pour sauter cette étape (plus rapide, colonne `image_urls` vide).
⚠️ Sur Colab, l'installation de `chromium-chromedriver` est tentée automatiquement. En local, assurez-vous d'avoir Google Chrome installé.


In [12]:
import re, time, html
from urllib.parse import quote

ACTIVER_SCRAPING_IMAGES = True   # -> False pour sauter cette étape
NB_IMAGES_PAR_PRODUIT = 3
TENTATIVES_PAR_PRODUIT = 2       # nb d'essais si aucune URL n'est trouvée au 1er coup

BLOCKED_DOMAINS = [
    "dreamstime.com", "shutterstock.com", "istockphoto.com", "alamy.com",
    "getty", "wikia.nocookie.net", "quizlet.com", "wikipedia.org",
    "pinterest.com", "tumblr.com",
]

# Bing encode le JSON interne dans plusieurs attributs selon la mise en page.
# On essaie plusieurs motifs, du plus fiable (murl) au plus permissif (src/data-src),
# pour éviter de se retrouver avec 0 résultat si Bing a légèrement changé son HTML.
PATTERNS_URLS = [
    r'"murl":"([^"]+)"',
    r"murl&quot;:&quot;([^&]+)&quot;",
    r'm="\{[^}]*?&quot;murl&quot;:&quot;([^&]+)&quot;',
    r'data-src="([^"]*?\.(?:jpg|jpeg|png|webp|gif)[^"]*)"',
    r'src="([^"]*?\.(?:jpg|jpeg|png|webp|gif)[^"]*)"',
]


def url_valide(url):
    if not url or "http" not in url:
        return False
    return not any(d in url.lower() for d in BLOCKED_DOMAINS)


def nettoyer_url(url):
    """Nettoie une URL d'image : décodage des slashs échappés + suppression des paramètres inutiles."""
    url = url.replace("\\/", "/")
    url = url.split("?")[0].split("&")[0]
    return url


class ScraperImages:
    """
    Recherche + extraction des URLs d'images.

    Points corrigés par rapport à la version précédente (qui ne trouvait aucune URL) :
    1) Anti-détection : Bing sert souvent une page vide / sans résultats aux navigateurs
       identifiés comme automatisés (headless). On masque le flag `navigator.webdriver`
       et on utilise une fenêtre de taille réaliste + un vrai user-agent.
    2) Attente explicite du chargement des vignettes (élément `.iusc`) au lieu d'un simple
       `time.sleep`, plus fiable sur connexion lente.
    3) Plusieurs motifs d'extraction (murl encodé en HTML-entities, murl brut, src/data-src)
       car Bing peut encoder le JSON différemment selon la page (mobile/desktop, A/B test).
    4) Décodage HTML (`html.unescape`) TOUJOURS appliqué avant la recherche, sinon les
       guillemets `&quot;` empêchent la regex de matcher (c'était la cause historique du 2e
       script de nettoyage séparé).
    5) Réessai automatique si 0 résultat au premier passage (page parfois non chargée à temps).
    6) Messages de diagnostic (taille de la page, nb de correspondances par motif) pour
       comprendre rapidement si Bing bloque ou si c'est un souci de motif.
    """

    def __init__(self):
        self.driver = None
        self.disponible = False
        try:
            if IN_COLAB:
                subprocess.run(["apt-get", "update", "-qq"], check=False)
                subprocess.run(["apt-get", "install", "-y", "-qq", "chromium-chromedriver"], check=False)

            from selenium import webdriver
            from selenium.webdriver.chrome.options import Options
            from selenium.webdriver.common.by import By
            from selenium.webdriver.support.ui import WebDriverWait
            from selenium.webdriver.support import expected_conditions as EC

            self.By = By
            self.WebDriverWait = WebDriverWait
            self.EC = EC

            options = Options()
            options.add_argument("--headless=new")
            options.add_argument("--no-sandbox")
            options.add_argument("--disable-dev-shm-usage")
            options.add_argument("--window-size=1920,1080")
            options.add_argument("--lang=fr-FR")
            # Anti-détection : Bing/Google bloquent souvent les navigateurs identifiés
            # comme automatisés (c'est la cause la plus fréquente d'un scraping qui
            # renvoie 0 URL alors que le code d'extraction est correct).
            options.add_argument("--disable-blink-features=AutomationControlled")
            options.add_experimental_option("excludeSwitches", ["enable-automation"])
            options.add_experimental_option("useAutomationExtension", False)
            options.add_argument(
                "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
            )

            try:
                from webdriver_manager.chrome import ChromeDriverManager
                from selenium.webdriver.chrome.service import Service
                self.driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
            except Exception:
                self.driver = webdriver.Chrome(options=options)

            # Masque le flag navigator.webdriver (détecté par Bing pour bloquer les bots)
            try:
                self.driver.execute_cdp_cmd(
                    "Page.addScriptToEvaluateOnNewDocument",
                    {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
                )
            except Exception:
                pass

            self.disponible = True
            print("✓ Selenium prêt (mode anti-détection activé)")
        except Exception as e:
            print(f"⚠️ Selenium indisponible ({e}) — la colonne image_urls sera laissée vide")

    def _extraire_urls_de_page(self, page_source, nb):
        page_decodee = html.unescape(page_source)
        urls = []
        for pattern in PATTERNS_URLS:
            matches = re.findall(pattern, page_decodee, re.IGNORECASE)
            for match in matches:
                u = nettoyer_url(match)
                if url_valide(u) and u not in urls:
                    urls.append(u)
                if len(urls) >= nb:
                    break
            if len(urls) >= nb:
                break
        return urls, page_decodee

    def recuperer_urls(self, nom_produit, nb=3, debug=False):
        if not self.disponible:
            return []
        nom_court = str(nom_produit).split("\n")[0][:80]
        url = f"https://www.bing.com/images/search?q={quote(nom_court)}&form=HDRSC2"

        for tentative in range(1, TENTATIVES_PAR_PRODUIT + 1):
            try:
                self.driver.get(url)

                # Attente explicite du chargement des vignettes (plus fiable qu'un sleep fixe)
                try:
                    self.WebDriverWait(self.driver, 8).until(
                        self.EC.presence_of_element_located((self.By.CSS_SELECTOR, ".iusc, .mimg"))
                    )
                except Exception:
                    time.sleep(2)  # fallback si le sélecteur n'apparaît jamais

                for _ in range(6):
                    self.driver.execute_script("window.scrollBy(0, window.innerHeight);")
                    time.sleep(0.4)

                urls, page_decodee = self._extraire_urls_de_page(self.driver.page_source, nb)

                if debug:
                    print(f"    [debug] tentative {tentative} — taille page: {len(page_decodee)} car. — "
                          f"'murl' présent: {'murl' in page_decodee} — URLs trouvées: {len(urls)}")

                if urls:
                    return urls[:nb]

                time.sleep(1.5)  # petite pause avant de réessayer
            except Exception as e:
                if debug:
                    print(f"    [debug] erreur tentative {tentative}: {e}")
                continue

        return []

    def fermer(self):
        if self.driver:
            try:
                self.driver.quit()
            except Exception:
                pass


if ACTIVER_SCRAPING_IMAGES:
    scraper = ScraperImages()
    liste_urls = []
    nb_avec_images = 0
    try:
        df_clean = df_clean.reset_index(drop=True)
        for idx, row in df_clean.iterrows():
            # debug=True sur les 3 premiers produits pour diagnostiquer rapidement
            # si le scraping échoue encore (regarder les messages [debug] ci-dessous).
            urls = scraper.recuperer_urls(
                row["NOM DU PRODUIT"], nb=NB_IMAGES_PAR_PRODUIT, debug=(idx < 3)
            )
            liste_urls.append(",".join(urls))
            if urls:
                nb_avec_images += 1
            if (idx + 1) % 20 == 0:
                print(f"   {idx + 1}/{len(df_clean)} produits traités... "
                      f"({nb_avec_images} avec au moins 1 image)")
    finally:
        scraper.fermer()

    df_clean["image_urls"] = liste_urls

    if nb_avec_images == 0:
        print("\n⚠️ Aucune image trouvée sur l'ensemble du fichier. Causes probables :")
        print("   - Bing bloque les requêtes automatisées depuis l'IP de Colab (fréquent) ;")
        print("   - le réseau/proxy de l'environnement empêche l'accès à bing.com ;")
        print("   - Bing a changé la structure de sa page de résultats.")
        print("   Regardez les lignes [debug] ci-dessus : si 'murl' présent: False, Bing ne")
        print("   sert pas la page attendue (blocage). Dans ce cas, il faut soit changer")
        print("   de méthode (API Bing Images officielle, ou une autre source d'images),")
        print("   soit relancer le scraping plus tard / depuis une autre IP.")
else:
    df_clean["image_urls"] = ""
    print("⚠️ Scraping images désactivé — colonne image_urls vide.")

print("\n✅ Étape scraping images terminée.")
display(df_clean[["NOM DU PRODUIT", "image_urls"]].head())


⚠️ Selenium indisponible (Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x55f6fefbd87a <unknown>
#1 0x55f6fe8efb99 <unknown>
#2 0x55f6fe92f11e <unknown>
#3 0x55f6fe92aa0f <unknown>
#4 0x55f6fe97afde <unknown>
#5 0x55f6fe97a69c <unknown>
#6 0x55f6fe93917b <unknown>
#7 0x55f6fe939f61 <unknown>
#8 0x55f6fef80247 <unknown>
#9 0x55f6fef7ea75 <unknown>
#10 0x55f6fef69b35 <unknown>
#11 0x55f6fef7f6ca <unknown>
#12 0x55f6fef52219 <unknown>
#13 0x55f6fefa7698 <unknown>
#14 0x55f6fefa7835 <unknown>
#15 0x55f6fefbc1c3 <unknown>
#16 0x7a02d8fbda83 <unknown>
) — la colonne image_urls sera laissée vide

⚠️ Aucune image trouvée sur l'ensemble du fichier. Causes probables :
   - Bing bloque les requêtes automatisées depuis l'IP de Colab (fréquent) ;
   - le réseau/proxy de l'env

,NOM DU PRODUIT,image_urls
0,Apple Clear Case with MagSafe for iPhone 12/12...,
1,Coque silicone Apple iPhone SE bleu abys,
2,Cque silicone Apple iPhone SE rose craie,
3,IPHONE 12/12 PRO case,
4,IPHONE SE case,


## 7) CELLULE 7 — Assemblage final et export Excel

Colonnes finales, dans l'ordre attendu :
`NOM DU PRODUIT, ean, sku, etat, quantite, id_tax_rules_group, idMarqueSite, idCategorySite, description, recap, price_particulier, price_profesionnel, image_urls, meta_title, meta_description, meta_keywords, mot_clés`


In [20]:
colonnes_finales = [
    "NOM DU PRODUIT", "EAN", "sku", "etat", "quantite", "id_tax_rules_group",
    "idMarqueSite", "idCategorySite", "description", "recap",
    "price_particulier", "price_profesionnel", "image_urls",
    "meta_title", "meta_description", "meta_keywords", "mot_clés", "actif",
]

df_final = df_clean[colonnes_finales].rename(columns={"EAN": "ean"}).copy()

print(f"✅ Dataset final prêt : {len(df_final)} produits, {len(df_final.columns)} colonnes")
display(df_final.head(10))

nom_sortie = "produits_finaux.xlsx"
sauvegarder_et_telecharger(df_final, nom_sortie)


✅ Dataset final prêt : 6 produits, 18 colonnes


,NOM DU PRODUIT,ean,sku,etat,quantite,id_tax_rules_group,idMarqueSite,idCategorySite,description,recap,price_particulier,price_profesionnel,image_urls,meta_title,meta_description,meta_keywords,mot_clés,actif
0,Apple Clear Case with MagSafe for iPhone 12/12...,0194252169575,NOP-0194252169575,Comme neuf,10,70,35,1889,<p><strong>Bonjour et bienvenue chez High-Tech...,"<table style=""border:1px solid #ddd;border-col...",49.17,49.17,,Apple Clear Case with MagSafe for iPhone 12/12...,Apple Clear Case with MagSafe for iPhone 12/12...,"apple, clear, case, with, magsafe, for, iphone...","apple, clear, case, with, magsafe, for, iphone...",1
1,Coque silicone Apple iPhone SE bleu abys,0194253035466,NOP-0194253035466,Comme neuf,2,70,35,1889,<p><strong>Bonjour et bienvenue chez High-Tech...,"<table style=""border:1px solid #ddd;border-col...",49.17,49.17,,Coque silicone Apple iPhone SE bleu abys,Coque silicone Apple iPhone SE bleu abys - Com...,"coque, silicone, apple, iphone, se, bleu, abys","coque, silicone, apple, iphone, se, bleu, abys",1
2,Cque silicone Apple iPhone SE rose craie,0194253035497,NOP-0194253035497,Comme neuf,1,70,35,1889,<p><strong>Bonjour et bienvenue chez High-Tech...,"<table style=""border:1px solid #ddd;border-col...",49.17,49.17,,Cque silicone Apple iPhone SE rose craie,Cque silicone Apple iPhone SE rose craie - Com...,"cque, silicone, apple, iphone, se, rose, craie","cque, silicone, apple, iphone, se, rose, craie",1
3,IPHONE 12/12 PRO case,0194252168295,NOP-0194252168295,Comme neuf,1,70,35,316,<p><strong>Bonjour et bienvenue chez High-Tech...,"<table style=""border:1px solid #ddd;border-col...",49.17,49.17,,IPHONE 12/12 PRO case,IPHONE 12/12 PRO case - Comme neuf,"iphone, 12/12, pro, case","iphone, 12/12, pro, case",1
4,IPHONE SE case,0194253035497,NOP-0194253035497,Comme neuf,1,70,35,1889,<p><strong>Bonjour et bienvenue chez High-Tech...,"<table style=""border:1px solid #ddd;border-col...",32.50,32.50,,IPHONE SE case,IPHONE SE case - Comme neuf,"iphone, se, case","iphone, se, case",1
5,iphone 12/12pro silicone case deep navy,0194252169056,NOP-0194252169056,Comme neuf,1,70,35,1889,<p><strong>Bonjour et bienvenue chez High-Tech...,"<table style=""border:1px solid #ddd;border-col...",49.17,49.17,,iphone 12/12pro silicone case deep navy,iphone 12/12pro silicone case deep navy - Comm...,"iphone, 12/12pro, silicone, case, deep, navy","iphone, 12/12pro, silicone, case, deep, navy",1


✅ Fichier sauvegardé : produits_finaux.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'produits_finaux.xlsx'

## 8) CELLULE TEST — Vérification rapide sur un échantillon (sans upload)

Cette cellule rejoue les étapes 1 et 2 sur un **petit échantillon inline** (issu de votre exemple), pour valider
rapidement la logique de nettoyage/fusion/EAN/longueur **sans avoir besoin d'uploader un fichier**.
Exécutez-la indépendamment du reste du notebook.


In [21]:
import pandas as pd

echantillon_test = pd.DataFrame({
    "NOM DU PRODUIT": [
        "TABLETTE GRAPHIQUE Artist 16 (2nd Gen)",
        "40*10mm ventilateur nf-a4x10 flx",
        "EMTEC X210 - SSD - 512 Go - externe (portable) - USB 3,2 Gen 2 (USB-C connecteur)",
        "MOTHERBOARD AMD Socket AM4 A520M-K V2",
        "Samsung Galaxy Book5 Pro 360",
        "casque LOGITECH G332",
        "AMIIBO",
        "AMIIBO",  # doublon volontaire pour tester la fusion
    ],
    "EAN": [
        "850032692168", "471623314691", "3126170173751", "4719331852771",
        "8806097102663", "5099206081963", "45496594183", "45496594183",
    ],
    "etat": ["NSB", "NSB", "NSB", "NOP", "nsb", "nsb", "NSB", "NSB"],
    "pc": ["369,99", "20,95", "233,99", "69,99", "1899", "59,99", "35", "35"],
    "prixVente": ["369,99", "20,95", "233,99", "69,99", "1899", "59,99", "35", "35"],
    "quantite": [2, 1, 4, 1, 17, 7, 10, 4],
})

print("--- Échantillon brut ---")
display(echantillon_test)

# --- Nettoyage (identique à la Cellule 1) ---
df_t = echantillon_test.copy()
df_t["NOM DU PRODUIT"] = df_t["NOM DU PRODUIT"].astype(str).str.strip()
df_t["etat"] = df_t["etat"].astype(str).str.strip().str.upper()
df_t["actif"] = 1
for col in ["pc", "prixVente"]:
    df_t[col] = df_t[col].astype(str).str.replace(",", ".", regex=False)
    df_t[col] = pd.to_numeric(df_t[col], errors="coerce")
df_t["quantite"] = pd.to_numeric(df_t["quantite"], errors="coerce").fillna(0)

df_t_clean = (
    df_t.groupby(["NOM DU PRODUIT", "EAN", "etat", "pc", "prixVente"], as_index=False)
        .agg({"quantite": "sum"})
)

print(f"\nLignes avant fusion : {len(df_t)} | après fusion : {len(df_t_clean)} (les 2 lignes AMIIBO doivent être fusionnées → quantité 14)")

# --- Vérification EAN + longueur (identique à la Cellule 2) ---
df_t_clean, *_ = corriger_ean(df_t_clean, colonne="EAN")
df_t_clean, _ = nettoyer_longueur_champs(df_t_clean, colonnes=["NOM DU PRODUIT"])

print("\n--- Résultat après nettoyage + fusion + vérification EAN ---")
display(df_t_clean)


--- Échantillon brut ---


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite
0,TABLETTE GRAPHIQUE Artist 16 (2nd Gen),850032692168,NSB,"369,99","369,99",2
1,40*10mm ventilateur nf-a4x10 flx,471623314691,NSB,"20,95","20,95",1
2,EMTEC X210 - SSD - 512 Go - externe (portable)...,3126170173751,NSB,"233,99","233,99",4
3,MOTHERBOARD AMD Socket AM4 A520M-K V2,4719331852771,NOP,"69,99","69,99",1
4,Samsung Galaxy Book5 Pro 360,8806097102663,nsb,1899,1899,17
5,casque LOGITECH G332,5099206081963,nsb,"59,99","59,99",7
6,AMIIBO,45496594183,NSB,35,35,10
7,AMIIBO,45496594183,NSB,35,35,4



Lignes avant fusion : 8 | après fusion : 7 (les 2 lignes AMIIBO doivent être fusionnées → quantité 14)
✅ EAN — complétés à 13 chiffres : 3 | anomalies (>13 chiffres, à vérifier) : 0
✅ Longueur des champs (max 128 caractères) :
   - NOM DU PRODUIT : 0 valeur(s) tronquée(s)

--- Résultat après nettoyage + fusion + vérification EAN ---


,NOM DU PRODUIT,EAN,etat,pc,prixVente,quantite
0,40*10mm ventilateur nf-a4x10 flx,0471623314691,NSB,20.95,20.95,1
1,AMIIBO,0045496594183,NSB,35.00,35.00,14
2,EMTEC X210 - SSD - 512 Go - externe (portable)...,3126170173751,NSB,233.99,233.99,4
3,MOTHERBOARD AMD Socket AM4 A520M-K V2,4719331852771,NOP,69.99,69.99,1
4,Samsung Galaxy Book5 Pro 360,8806097102663,NSB,1899.00,1899.00,17
5,TABLETTE GRAPHIQUE Artist 16 (2nd Gen),0850032692168,NSB,369.99,369.99,2
6,casque LOGITECH G332,5099206081963,NSB,59.99,59.99,7
